# P/D 调度时间线
读取 HTML 使用的同一份缓存；这里的概览为小时聚合，不能当作逐轮瞬时 BS。

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

base = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/dashboard_timelines.py').exists())
cache_root = base / 'analysis/scheduler_timelines/assets'
run = sorted(p for p in cache_root.iterdir() if p.is_dir())[-1]
cases = sorted(run.glob('*.json'))
CASE = cases[0]  # 可改成其他实验的缓存路径
data = json.loads(CASE.read_text())
print(data['case']['label'])
print(data['source_directory'])


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, stream in zip(axes, data['overview']):
    frame = pd.DataFrame(stream['rows'], columns=data['overview_columns'])
    days = (frame['hour'] + 0.5) / 24
    ax.plot(days, frame['max_bs'], label='hourly peak BS')
    ax.plot(days, frame['active_time_weighted_mean_bs'], label='compute-time-weighted BS', alpha=0.7)
    ax.set_title(stream['identity']['role'])
    ax.set_ylim(bottom=0)
    ax.set_ylabel('requests')
    ax.legend()
axes[-1].set_xlabel('simulation days')
fig.tight_layout()
plt.show()
